# PLINK / GWAS QC Workflow Agent
## Colab 04 — Command Generator + Command Validator Loop

這是期末專案第四階段。

流程：

```text
Verified QC Plan
↓
Groq Command Generator
↓
PLINK Commands
↓
Python Command Validator
↓
FAIL → Groq 修正
↓
Recheck
↓
PASS
```

### 本階段重點
- LLM 負責把已通過的 QC Plan 轉成 PLINK commands
- Python Validator 負責檢查：
  - unknown PLINK flags
  - `--bfile`
  - `--out`
  - step 對應 flag
  - `--pheno-name` / `--pheno`
  - `--covar-name` / `--covar`
  - case-control / `--logistic`
  - final GWAS 不可誤用 LD-pruned subset
- 最後展示 `FAIL → 修正 → PASS`

## 0. Colab Secret

請確認 Colab Secret：

```text
Groq
```

已開啟「筆記本存取權」。

本 Notebook 不會印出 API Key。

## 1. 安裝套件

In [ ]:
!pip -q install groq

## 2. Imports + Groq Client

In [ ]:
import re
import json
import shlex
from copy import deepcopy
from pprint import pprint

from groq import Groq, BadRequestError
from google.colab import userdata

GROQ_API_KEY = userdata.get("Groq")

if not GROQ_API_KEY:
    raise ValueError("找不到 Colab Secret：Groq")

client = Groq(api_key=GROQ_API_KEY)

MODEL_NAME = "openai/gpt-oss-20b"
MAX_COMMAND_ITERATIONS = 3
SEED = 123

print("Groq client: READY")

## 3. Verified QC Plan

如果你在同一個 runtime 裡先跑完 03，
程式會優先讀取：

```python
state["qc_plan"]
```

如果是新的 runtime，則使用下方內建的 verified plan，
讓這份 Notebook 可以獨立執行。

In [ ]:
DEFAULT_VERIFIED_QC_PLAN = [
    {
        "step": 1,
        "name": "Load PLINK Binary Files",
        "purpose": "Load the PLINK BED/BIM/FAM dataset.",
        "plink_flag": "--bfile",
        "example_setting": "study",
        "note": "Input step.",
        "source": "S01"
    },
    {
        "step": 2,
        "name": "Variant Missingness",
        "purpose": "Remove variants with excessive missing genotype calls.",
        "plink_flag": "--geno",
        "example_setting": "0.02",
        "note": "Example threshold only.",
        "source": "S02"
    },
    {
        "step": 3,
        "name": "Sample Missingness",
        "purpose": "Remove samples with excessive missing genotype calls.",
        "plink_flag": "--mind",
        "example_setting": "0.02",
        "note": "Example threshold only.",
        "source": "S02"
    },
    {
        "step": 4,
        "name": "MAF Filtering",
        "purpose": "Filter variants by minor allele frequency.",
        "plink_flag": "--maf",
        "example_setting": "0.01",
        "note": "Example threshold only.",
        "source": "S02"
    },
    {
        "step": 5,
        "name": "HWE Filtering",
        "purpose": "Filter variants using Hardy-Weinberg equilibrium test.",
        "plink_flag": "--hwe",
        "example_setting": "1e-6",
        "note": "Example threshold only.",
        "source": "S02"
    },
    {
        "step": 6,
        "name": "LD Pruning",
        "purpose": "Create an approximately LD-independent SNP subset for PCA/relatedness.",
        "plink_flag": "--indep-pairwise",
        "example_setting": "50 5 0.2",
        "note": "Do not automatically use the pruned subset as the final GWAS dataset.",
        "source": "S03"
    },
    {
        "step": 7,
        "name": "Relatedness",
        "purpose": "Estimate pairwise relatedness / IBD.",
        "plink_flag": "--genome",
        "example_setting": None,
        "note": "Use LD-pruned variants.",
        "source": "S04"
    },
    {
        "step": 8,
        "name": "PCA",
        "purpose": "Estimate population structure.",
        "plink_flag": "--pca",
        "example_setting": "10",
        "note": "Number of PCs is study-specific.",
        "source": "S05"
    },
    {
        "step": 9,
        "name": "Association Analysis",
        "purpose": "Run case-control logistic association with covariates.",
        "plink_flag": "--logistic",
        "example_setting": None,
        "note": "Use the post-QC association dataset, not only the LD-pruned subset.",
        "source": "S06, S08"
    }
]

study_info = {
    "study_design": "case-control",
    "data_format": "plink_binary",
    "sample_size": 3000,
    "phenotype": "Disease Status",
    "covariates": ["Age", "Sex"],
    "need_pca": True,
    "need_relatedness": True,
    "association": "logistic"
}

try:
    if "state" in globals() and state.get("qc_plan"):
        verified_qc_plan = deepcopy(state["qc_plan"])
        print("Using QC plan from previous notebook state.")
    else:
        verified_qc_plan = deepcopy(DEFAULT_VERIFIED_QC_PLAN)
        print("Using built-in verified QC plan.")
except Exception:
    verified_qc_plan = deepcopy(DEFAULT_VERIFIED_QC_PLAN)
    print("Using built-in verified QC plan.")

pprint(verified_qc_plan)

## 4. Command Output Schema

每個 QC step 可有一個或多個 command。

例如 LD pruning 可能需要：

```text
--indep-pairwise
```

以及建立 pruned dataset 的：

```text
--extract
```

In [ ]:
COMMAND_SCHEMA = {
    "type": "object",
    "properties": {
        "command_plan": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "step": {"type": "integer"},
                    "step_name": {"type": "string"},
                    "commands": {
                        "type": "array",
                        "items": {"type": "string"}
                    },
                    "purpose": {"type": "string"},
                    "source": {"type": "string"}
                },
                "required": [
                    "step",
                    "step_name",
                    "commands",
                    "purpose",
                    "source"
                ],
                "additionalProperties": False
            }
        }
    },
    "required": ["command_plan"],
    "additionalProperties": False
}

## 5. Command Generator Prompt

重點：
- 不 invent flag
- 每個 command 必須有 `--bfile` 與 `--out`
- association 使用 post-QC dataset
- LD-pruned dataset 只給 Relatedness / PCA 使用
- phenotype / covariate 使用 Wiki 已支撐的格式

In [ ]:
COMMAND_SYSTEM_PROMPT = '''
You are the PLINK command-generation component of a GWAS QC Workflow Agent.

Convert a VERIFIED QC PLAN into executable-looking PLINK 1.9 command examples.

STRICT RULES:
- Follow the verified QC plan.
- Do not invent PLINK flags.
- Every command must begin with "plink".
- Every command must include --bfile and --out.
- Use --make-bed when a filtering step should create a new PLINK binary dataset.
- Use example thresholds from the verified plan only as examples.
- LD pruning may create a pruned subset for Relatedness and PCA.
- The final association analysis must use the post-QC association dataset, NOT only the LD-pruned subset.
- For case-control association use --logistic.
- When using --pheno-name, also include --pheno.
- When using --covar-name, also include --covar.
- If covariates are supplied in study information, include them in association analysis.
- Preserve Source IDs from the verified QC plan.
- Return only the requested structured JSON.
'''.strip()

## 6. Robust Groq JSON Helper

先使用 strict JSON Schema。
若 Groq 偶發回 400，再 fallback 到 JSON Object Mode，
最後仍由 Python 做本地 shape check。

In [ ]:
REQUIRED_COMMAND_KEYS = {
    "step",
    "step_name",
    "commands",
    "purpose",
    "source"
}

def validate_command_json_shape(data):
    errors = []

    if not isinstance(data, dict):
        return ["Top-level response must be a JSON object."]

    if "command_plan" not in data:
        return ["Missing top-level key: command_plan."]

    plan = data["command_plan"]

    if not isinstance(plan, list) or len(plan) == 0:
        return ["command_plan must be a non-empty list."]

    for i, item in enumerate(plan, start=1):
        if not isinstance(item, dict):
            errors.append(f"Item {i} must be a JSON object.")
            continue

        missing = REQUIRED_COMMAND_KEYS - set(item.keys())
        if missing:
            errors.append(
                f"Item {i} missing keys: {sorted(missing)}"
            )

        if "commands" in item:
            if (
                not isinstance(item["commands"], list)
                or len(item["commands"]) == 0
            ):
                errors.append(
                    f"Item {i}: commands must be a non-empty list."
                )

    return errors


def call_command_json(
    user_prompt,
    schema_name
):
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": COMMAND_SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],
            temperature=0.0,
            seed=SEED,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": schema_name,
                    "strict": True,
                    "schema": COMMAND_SCHEMA
                }
            }
        )

        data = json.loads(
            response.choices[0].message.content
        )

        shape_errors = validate_command_json_shape(data)

        if not shape_errors:
            print("Groq output mode: STRICT JSON SCHEMA")
            return data

    except BadRequestError:
        print(
            "Strict Structured Output failed. "
            "Fallback: JSON Object Mode"
        )

    fallback_prompt = (
        user_prompt
        + "\n\nReturn ONLY valid JSON. "
          "Top-level key must be command_plan. "
          "Each item must contain step, step_name, commands, purpose, source. "
          "commands must be a non-empty array of strings."
    )

    for attempt in range(1, 3):
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": COMMAND_SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": fallback_prompt
                }
            ],
            temperature=0.0,
            response_format={"type": "json_object"}
        )

        data = json.loads(
            response.choices[0].message.content
        )

        shape_errors = validate_command_json_shape(data)

        if not shape_errors:
            print(
                f"Groq output mode: JSON OBJECT "
                f"(fallback attempt {attempt})"
            )
            return data

        fallback_prompt += (
            "\nYour previous JSON failed these checks:\n"
            + "\n".join(f"- {x}" for x in shape_errors)
        )

    raise RuntimeError(
        "Groq could not produce usable command JSON."
    )

## 7. Generate PLINK Commands

In [ ]:
def generate_commands(
    study_info,
    verified_qc_plan
):
    prompt = (
        "STUDY INFORMATION\n"
        + json.dumps(
            study_info,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nVERIFIED QC PLAN\n"
        + json.dumps(
            verified_qc_plan,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nGenerate PLINK commands for the verified plan."
    )

    data = call_command_json(
        prompt,
        schema_name="plink_command_plan"
    )

    return data["command_plan"]


command_plan = generate_commands(
    study_info,
    verified_qc_plan
)

pprint(command_plan)

# Part B — Command Validator

## 8. Allowed PLINK Flags

第一版只允許 Wiki / 本專案已使用的 flags。

In [ ]:
ALLOWED_FLAGS = {
    "--bfile",
    "--geno",
    "--mind",
    "--maf",
    "--hwe",
    "--indep-pairwise",
    "--extract",
    "--genome",
    "--pca",
    "--pheno",
    "--pheno-name",
    "--covar",
    "--covar-name",
    "--logistic",
    "--1",
    "--make-bed",
    "--out"
}

print("Allowed flags:", len(ALLOWED_FLAGS))

## 9. Command Parsing Helpers

In [ ]:
def normalize_command(command):
    command = command.replace("\\\n", " ")
    command = re.sub(r"\s+", " ", command).strip()
    return command


def tokenize_command(command):
    command = normalize_command(command)

    try:
        return shlex.split(command)
    except Exception:
        return command.split()


def get_flags(command):
    tokens = tokenize_command(command)
    return [t for t in tokens if t.startswith("--")]


def get_flag_value(command, flag):
    tokens = tokenize_command(command)

    if flag not in tokens:
        return None

    idx = tokens.index(flag)

    if idx + 1 >= len(tokens):
        return None

    nxt = tokens[idx + 1]

    if nxt.startswith("--"):
        return None

    return nxt

## 10. Command Validator

檢查四層：

### A. 基本結構
- 必須以 `plink` 開頭
- 每條 command 有 `--bfile`
- 每條 command 有 `--out`

### B. Unknown Flags
- 不允許 hallucinated flags

### C. Step-specific Flags
例如：
- Variant Missingness → `--geno`
- PCA → `--pca`
- Association → `--logistic`

### D. Dependencies / Logic
- `--pheno-name` → `--pheno`
- `--covar-name` → `--covar`
- case-control → association 需 `--logistic`
- association `--bfile` 不可明顯指向 `ld_pruned`

In [ ]:
def validate_command_plan(
    study_info,
    verified_qc_plan,
    command_plan
):
    errors = []

    if not isinstance(command_plan, list) or len(command_plan) == 0:
        return ["Command plan is empty."]

    expected_step_flags = {
        "load plink binary files": "--bfile",
        "variant missingness": "--geno",
        "sample missingness": "--mind",
        "maf filtering": "--maf",
        "hwe filtering": "--hwe",
        "ld pruning": "--indep-pairwise",
        "relatedness": "--genome",
        "pca": "--pca",
        "association analysis": "--logistic"
    }

    # QC plan source map
    plan_source_map = {
        str(x.get("name", "")).strip().lower():
        str(x.get("source", "")).strip()
        for x in verified_qc_plan
    }

    for item_idx, item in enumerate(command_plan, start=1):
        step_name = str(
            item.get("step_name", "")
        ).strip()

        step_key = step_name.lower()
        commands = item.get("commands", [])
        item_source = str(
            item.get("source", "")
        ).strip()

        if not commands:
            errors.append(
                f"Step {item_idx} ({step_name}) has no command."
            )
            continue

        # source consistency
        matching_plan_source = None

        for plan_name, source in plan_source_map.items():
            if (
                step_key in plan_name
                or plan_name in step_key
            ):
                matching_plan_source = source
                break

        if (
            matching_plan_source
            and item_source != matching_plan_source
        ):
            errors.append(
                f"Step {item_idx} ({step_name}) source does not match "
                f"verified QC plan: expected {matching_plan_source}, "
                f"got {item_source}."
            )

        all_flags = []

        for cmd_idx, command in enumerate(commands, start=1):
            cmd = normalize_command(command)
            tokens = tokenize_command(cmd)

            if not tokens:
                errors.append(
                    f"Step {item_idx}, command {cmd_idx} is empty."
                )
                continue

            if tokens[0].lower() != "plink":
                errors.append(
                    f"Step {item_idx}, command {cmd_idx} "
                    "must begin with 'plink'."
                )

            flags = get_flags(cmd)
            all_flags.extend(flags)

            # unknown flags
            for flag in flags:
                if flag not in ALLOWED_FLAGS:
                    errors.append(
                        f"Step {item_idx}, command {cmd_idx}: "
                        f"Unknown PLINK flag: {flag}"
                    )

            # common required flags
            if "--bfile" not in flags:
                errors.append(
                    f"Step {item_idx}, command {cmd_idx}: "
                    "Missing --bfile."
                )

            if "--out" not in flags:
                errors.append(
                    f"Step {item_idx}, command {cmd_idx}: "
                    "Missing --out."
                )

            # dependencies
            if (
                "--pheno-name" in flags
                and "--pheno" not in flags
            ):
                errors.append(
                    f"Step {item_idx}, command {cmd_idx}: "
                    "--pheno-name requires --pheno."
                )

            if (
                "--covar-name" in flags
                and "--covar" not in flags
            ):
                errors.append(
                    f"Step {item_idx}, command {cmd_idx}: "
                    "--covar-name requires --covar."
                )

        # step-specific expected flag
        expected = None

        for key, flag in expected_step_flags.items():
            if key in step_key:
                expected = flag
                break

        if (
            expected
            and expected not in all_flags
        ):
            errors.append(
                f"Step {item_idx} ({step_name}) "
                f"is missing expected flag {expected}."
            )

        # Association-specific checks
        if "association" in step_key:
            association_commands = [
                normalize_command(c)
                for c in commands
            ]

            combined_flags = set()

            for cmd in association_commands:
                combined_flags.update(
                    get_flags(cmd)
                )

                bfile_value = get_flag_value(
                    cmd,
                    "--bfile"
                )

                if (
                    bfile_value
                    and (
                        "prune" in bfile_value.lower()
                        or "ld_" in bfile_value.lower()
                    )
                ):
                    errors.append(
                        f"Step {item_idx} (Association Analysis) "
                        "appears to use an LD-pruned dataset as the "
                        "final GWAS input."
                    )

            if (
                study_info.get("study_design") == "case-control"
                and study_info.get("association") == "logistic"
                and "--logistic" not in combined_flags
            ):
                errors.append(
                    f"Step {item_idx} (Association Analysis) "
                    "requires --logistic for this study."
                )

            if (
                study_info.get("covariates")
                and "--covar" not in combined_flags
            ):
                errors.append(
                    f"Step {item_idx} (Association Analysis) "
                    "has study covariates but no --covar."
                )

    return errors

## 11. Validate Initial Command Plan

In [ ]:
command_errors = validate_command_plan(
    study_info,
    verified_qc_plan,
    command_plan
)

print(
    "COMMAND VALIDATION:",
    "PASS"
    if not command_errors
    else "FAIL"
)

for i, error in enumerate(
    command_errors,
    start=1
):
    print(f"{i}. {error}")

# Part C — Automatic Command Revision Loop

## 12. Revision Function

In [ ]:
def revise_commands(
    study_info,
    verified_qc_plan,
    current_command_plan,
    errors
):
    prompt = (
        "STUDY INFORMATION\n"
        + json.dumps(
            study_info,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nVERIFIED QC PLAN\n"
        + json.dumps(
            verified_qc_plan,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nCURRENT COMMAND PLAN\n"
        + json.dumps(
            current_command_plan,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nCOMMAND VALIDATOR ERRORS\n"
        + json.dumps(
            errors,
            ensure_ascii=False,
            indent=2
        )
        + "\n\nPatch the command plan minimally. "
          "Fix every validator error. "
          "Do not remove unrelated valid commands. "
          "Do not invent flags. "
          "Return the full corrected command plan."
    )

    data = call_command_json(
        prompt,
        schema_name="revised_plink_command_plan"
    )

    return data["command_plan"]

## 13. Run Command Loop

In [ ]:
command_state = {
    "command_plan": deepcopy(command_plan),
    "command_errors": [],
    "command_iteration": 0
}

while True:
    command_state["command_errors"] = validate_command_plan(
        study_info,
        verified_qc_plan,
        command_state["command_plan"]
    )

    print("=" * 70)
    print(
        f'Command validation round '
        f'{command_state["command_iteration"] + 1}'
    )

    if not command_state["command_errors"]:
        print("COMMAND VALIDATION: PASS")
        break

    print("COMMAND VALIDATION: FAIL")

    for i, error in enumerate(
        command_state["command_errors"],
        start=1
    ):
        print(f"{i}. {error}")

    if (
        command_state["command_iteration"]
        >= MAX_COMMAND_ITERATIONS
    ):
        print("\nSTOP / NEED HUMAN REVIEW")
        break

    command_state["command_plan"] = revise_commands(
        study_info,
        verified_qc_plan,
        command_state["command_plan"],
        command_state["command_errors"]
    )

    command_state["command_iteration"] += 1

print(
    "\nFinal command revision count:",
    command_state["command_iteration"]
)

## 14. Final Command Plan

In [ ]:
print(
    json.dumps(
        command_state["command_plan"],
        ensure_ascii=False,
        indent=2
    )
)

# Part D — Demo：故意製造 Command Errors

故意做兩個錯誤：

1. 加入不存在的：
   `--remove-bad-snps`
2. Association command 移除 `--covar`

預期：

```text
DEMO COMMAND VALIDATION: FAIL
```

In [ ]:
demo_bad_commands = deepcopy(
    command_state["command_plan"]
)

# 找第一條 command，注入 hallucinated flag
injected = False

for item in demo_bad_commands:
    if item.get("commands") and not injected:
        item["commands"][0] = (
            item["commands"][0]
            + " --remove-bad-snps"
        )
        injected = True

# Association 移除 --covar 與其檔名
for item in demo_bad_commands:
    if "association" in str(
        item.get("step_name", "")
    ).lower():
        new_commands = []

        for cmd in item["commands"]:
            tokens = tokenize_command(cmd)

            cleaned = []
            skip_next = False

            for token in tokens:
                if skip_next:
                    skip_next = False
                    continue

                if token == "--covar":
                    skip_next = True
                    continue

                cleaned.append(token)

            new_commands.append(
                " ".join(cleaned)
            )

        item["commands"] = new_commands


demo_command_errors = validate_command_plan(
    study_info,
    verified_qc_plan,
    demo_bad_commands
)

print(
    "DEMO COMMAND VALIDATION:",
    "FAIL"
    if demo_command_errors
    else "PASS"
)

for i, error in enumerate(
    demo_command_errors,
    start=1
):
    print(f"{i}. {error}")

## 15. Groq 自動修正 Demo

In [ ]:
demo_revised_commands = revise_commands(
    study_info,
    verified_qc_plan,
    demo_bad_commands,
    demo_command_errors
)

demo_command_recheck = validate_command_plan(
    study_info,
    verified_qc_plan,
    demo_revised_commands
)

print(
    "AFTER COMMAND REVISION:",
    "PASS"
    if not demo_command_recheck
    else "FAIL"
)

for error in demo_command_recheck:
    print("-", error)

# 16. 第四階段完成標準

理想輸出：

```text
COMMAND VALIDATION: PASS
```

如果第一次沒過，也可以：

```text
Command validation round 1
FAIL

Command validation round 2
PASS
```

最後 Demo 應該看到：

```text
DEMO COMMAND VALIDATION: FAIL
1. Unknown PLINK flag: --remove-bad-snps
2. Association ... no --covar

AFTER COMMAND REVISION: PASS
```

---

# 完成 04 後

兩個主要 Agent Loop 就都有了：

```text
RAG
↓
QC Planning
↓
Plan Validator
↓
FAIL → Revise
↓
PASS
↓
Command Generator
↓
Command Validator
↓
FAIL → Revise
↓
PASS
```

下一階段會做：

**05 — Final Report + Gradio UI**

把所有結果整合成使用者可以直接操作的 Web App。